# EEGNet: Minimal Train & Test (Colab-ready)
This notebook loads labeled EEG segments, prepares splits, trains EEGNet, and evaluates it.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
# Colab deps (safe no-op locally)
import sys
if 'google.colab' in sys.modules:
    %pip install -q tensorflow scikit-learn

In [ ]:
# All imports
import os
import sys
import math
import numpy as np
import pandas as pd
from pathlib import Path
from typing import List, Sequence, Tuple, Optional, Union

# Machine Learning
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score

# TensorFlow/Keras
import tensorflow as tf
from tensorflow.keras import utils as np_utils
from tensorflow.keras.models import Model, Sequential
from tensorflow.keras.layers import (
    Dense, Activation, Dropout, Conv2D, AveragePooling2D, MaxPooling2D,
    SeparableConv2D, DepthwiseConv2D, BatchNormalization, SpatialDropout2D,
    Input, Flatten
)
from tensorflow.keras.constraints import max_norm
from tensorflow.keras.callbacks import EarlyStopping

In [ ]:
tf.config.list_physical_devices('GPU')

In [ ]:
!nvidia-smi

In [ ]:
# Paths & basic config (edit BASE_DIR if running on Colab)
BASE_DIR = '/content/drive/MyDrive/'
LABELED_EEG_DATA_DIR = Path(BASE_DIR) / 'labeled_eeg_data'
PRE_WINDOW_MS = 500
POST_WINDOW_MS = 500

# --------------------------
# CHANNEL CONFIGURATION
# --------------------------
# Auto-discover available subjects

# Option 1: Minimal channels (fast, low RAM)
# (define CHANNELS_MINIMAL if needed)

# Option 2: Standard channels (more coverage, more RAM)
DEFAULT_EEGNET_CHANNELS = ['F3', 'F4', 'F7', 'F8', 'Fz', 'FC1', 'FC2', 'FC5', 'FC6', 
                          'Fp1', 'Fp2', 'Pz', 'P3', 'P4', 'P7', 'P8']


In [ ]:
# --------------------------
# AUTO-DISCOVER SUBJECTS & SET CHANNELS
# --------------------------
# Find all subject folders (starting with 'P')
USE_SUBJECTS = sorted([
    d.name for d in Path(LABELED_EEG_DATA_DIR).iterdir()
    if d.is_dir() and d.name.startswith('P')
])
print(f"Found {len(USE_SUBJECTS)} subjects: {USE_SUBJECTS[:10]} ...")

# Set channels to use (edit as needed)
CHANNELS_TO_USE = [
    'F3', 'F4', 'F7', 'F8', 'Fz', 'FC1', 'FC2', 'FC5', 'FC6',
    'Fp1', 'Fp2', 'Pz', 'P3', 'P4', 'P7', 'P8'
]
print(f"Using {len(CHANNELS_TO_USE)} channels: {CHANNELS_TO_USE}")


In [ ]:
# --------------------------
# SIMPLE EEG DATA LOADING
# --------------------------
def load_eeg_data(subjects, channels, pre_ms, post_ms):
    """Load EEG data for specified subjects and channels."""
    print(f"Loading {len(subjects)} subjects with {len(channels)} channels...")
    
    all_X, all_y = [], []
    
    for subject in subjects:
        file_path = LABELED_EEG_DATA_DIR / subject / f"{pre_ms}_{post_ms}_training_data.npz"
        if not file_path.exists():
            continue
            
        # Load with memory mapping first to check channels
        with np.load(file_path, mmap_mode='r') as data:
            ch_names = data['channel_names']
            
            # Find indices of channels we want
            ch_indices = []
            for ch in channels:
                if ch in ch_names:
                    ch_indices.append(list(ch_names).index(ch))
            
            if len(ch_indices) == 0:
                continue  # Skip if no channels found
            
            # Only load the data we need - much more memory efficient
            X_full = data['X']  # Still memory mapped
            y = data['y'][:]    # Load labels (small)
            
            # Extract only the channels we want
            X_selected = X_full[:, ch_indices, :].copy()  # Copy only needed data
            
        # Now X_full is automatically freed when we exit the 'with' block
        
        # 🚨 CRITICAL: Check and apply scaling fix
        if abs(X_selected.mean()) < 1e-3:  # Data in Volts
            X_selected = X_selected * 1e6
        
        all_X.append(X_selected)
        all_y.append(y)
        
        # Force garbage collection to free memory
        del X_selected
    
    # Combine all subjects
    X_combined = np.concatenate(all_X, axis=0)
    y_combined = np.concatenate(all_y, axis=0)
    
    print(f"Loaded {X_combined.shape[0]} trials from {len(all_X)} subjects")
    print(f"Using {len(ch_indices)} channels, {X_combined.shape[2]} time points")
    return X_combined, y_combined

In [ ]:
# --------------------------
# DATA LOADING METHODS: INTRA- AND INTER-SUBJECT
# --------------------------
def load_data_intra():
    """INTRA-SUBJECT: Mixes all subjects, random trial split (can inflate performance)."""
    print("Loading INTRA-SUBJECT data (mixed subjects)...")
    X_all, y_all = load_eeg_data(USE_SUBJECTS, CHANNELS_TO_USE, PRE_WINDOW_MS, POST_WINDOW_MS)
    return prepare_datasets(X_all, y_all)

def load_data_inter():
    """INTER-SUBJECT: Separate subjects for train/val/test (proper generalization)."""
    n_total = len(USE_SUBJECTS)
    n_train = int(0.6 * n_total)
    n_val = int(0.2 * n_total)
    train_subj = USE_SUBJECTS[:n_train]
    val_subj = USE_SUBJECTS[n_train:n_train+n_val]
    test_subj = USE_SUBJECTS[n_train+n_val:]
    print(f"  Train subjects: {train_subj}")
    print(f"  Val subjects: {val_subj}")
    print(f"  Test subjects: {test_subj}")
    X_train, y_train = load_eeg_data(train_subj, CHANNELS_TO_USE, PRE_WINDOW_MS, POST_WINDOW_MS)
    X_val, y_val = load_eeg_data(val_subj, CHANNELS_TO_USE, PRE_WINDOW_MS, POST_WINDOW_MS)
    X_test, y_test = load_eeg_data(test_subj, CHANNELS_TO_USE, PRE_WINDOW_MS, POST_WINDOW_MS)
    # Prepare each set separately (no internal split, just normalization/reshaping)
    def norm_and_format(X, y):
        X = X.transpose(0, 1, 2)
        X = X.reshape(X.shape[0], X.shape[1], X.shape[2], 1)
        X_mean = np.mean(X)
        X_std = np.std(X)
        X = (X - X_mean) / (X_std + 1e-8)
        y = tf.keras.utils.to_categorical(y.astype(int), 2)
        print(f"Data normalized: mean={X_mean:.2f}, std={X_std:.2f}")
        print(f"After normalization: mean={np.mean(X):.4f}, std={np.std(X):.4f}")
        return X, y
    X_train, y_train = norm_and_format(X_train, y_train)
    X_val, y_val = norm_and_format(X_val, y_val)
    X_test, y_test = norm_and_format(X_test, y_test)
    return X_train, X_val, X_test, y_train, y_val, y_test


In [ ]:
# --------------------------
# SIMPLE DATA PREPARATION - With Normalization
# --------------------------
def prepare_datasets(X, y, test_size=0.2, val_size=0.2, batch_size=64, seed=42):
    """Simple train/val/test split with proper EEGNet formatting and normalization."""
    
    # Convert to EEGNet format: (samples, channels, time_points, 1)
    X = X.transpose(0, 1, 2)  # Already (trials, channels, samples)
    X = X.reshape(X.shape[0], X.shape[1], X.shape[2], 1)  # Add channel dimension
    
    # 🚨 CRITICAL: Standardize the data for neural networks
    # Calculate stats on the full dataset first
    X_mean = np.mean(X)
    X_std = np.std(X)
    X = (X - X_mean) / (X_std + 1e-8)  # Prevent division by zero
    
    # Convert labels to categorical
    y = tf.keras.utils.to_categorical(y.astype(int), 2)
    
    # Split data
    X_train, X_test, y_train, y_test = train_test_split(
        X, y, test_size=test_size, random_state=seed, stratify=np.argmax(y, axis=1)
    )
    
    X_train, X_val, y_train, y_val = train_test_split(
        X_train, y_train, test_size=val_size, random_state=seed, stratify=np.argmax(y_train, axis=1)
    )
    
    print(f"Data normalized: mean={X_mean:.2f}, std={X_std:.2f}")
    print(f"After normalization: mean={np.mean(X):.4f}, std={np.std(X):.4f}")
    
    return X_train, X_val, X_test, y_train, y_val, y_test

In [ ]:
# EEGNet model definition
def EEGNet(nb_classes, Chans, Samples, dropoutRate=0.2, kernLength=64, F1=8, D=3, F2=16, norm_rate=0.25, dropoutType='Dropout'):
    if dropoutType == 'SpatialDropout2D':
        dropoutType = SpatialDropout2D
    elif dropoutType == 'Dropout':
        dropoutType = Dropout
    else:
        raise ValueError('dropoutType must be SpatialDropout2D or Dropout')

    input1 = Input(shape=(Chans, Samples, 1))
    block1 = Conv2D(F1, (1, kernLength), padding='same', use_bias=False)(input1)
    block1 = BatchNormalization()(block1)
    block1 = DepthwiseConv2D((Chans, 1), use_bias=False, depth_multiplier=D, depthwise_constraint=max_norm(1.))(block1)
    block1 = BatchNormalization()(block1)
    block1 = Activation('elu')(block1)
    block1 = AveragePooling2D((1, 4))(block1)
    block1 = dropoutType(dropoutRate)(block1)

    block2 = SeparableConv2D(F2, (1, 16), use_bias=False, padding='same')(block1)
    block2 = BatchNormalization()(block2)
    block2 = Activation('elu')(block2)
    block2 = AveragePooling2D((1, 8))(block2)
    block2 = dropoutType(dropoutRate)(block2)

    flatten = Flatten(name='flatten')(block2)
    dense = Dense(nb_classes, name='dense', kernel_constraint=max_norm(norm_rate))(flatten)
    softmax = Activation('softmax', name='softmax')(dense)
    return Model(inputs=input1, outputs=softmax)

In [ ]:
# --------------------------
# RESULTS TRACKING
# --------------------------
results = {
    'intra': {'Model': [], 'Train': [], 'Val': [], 'Test': [], 'Gap': []},
    'inter': {'Model': [], 'Train': [], 'Val': [], 'Test': [], 'Gap': []}
}

def record(eval_method, model_name, train_acc, val_acc, test_acc):
    """Record model performance for comparison."""
    gap = train_acc - test_acc
    results[eval_method]['Model'].append(model_name)
    results[eval_method]['Train'].append(train_acc)
    results[eval_method]['Val'].append(val_acc)
    results[eval_method]['Test'].append(test_acc)
    results[eval_method]['Gap'].append(gap)
    print(f"  {model_name} ({eval_method}): Test {test_acc:.1%}, Gap {gap:.1%}")

In [ ]:
# 1. LOGISTIC REGRESSION
print("\n=== LOGISTIC REGRESSION ===")

for eval_method in ['intra', 'inter']:
    # Load data
    if eval_method == 'intra':
        # Combine all subjects, then split randomly (unified)
        X_all, y_all = load_eeg_data(USE_SUBJECTS, CHANNELS_TO_USE, PRE_WINDOW_MS, POST_WINDOW_MS)
        X_train, X_val, X_test, y_train, y_val, y_test = prepare_datasets(X_all, y_all)
    else:
        X_train, X_val, X_test, y_train, y_val, y_test = load_data_inter()
    
    # Prepare for sklearn
    X_train_flat = X_train.reshape(X_train.shape[0], -1)
    X_val_flat = X_val.reshape(X_val.shape[0], -1)
    X_test_flat = X_test.reshape(X_test.shape[0], -1)
    y_train_flat = np.argmax(y_train, axis=1)
    y_val_flat = np.argmax(y_val, axis=1)
    y_test_flat = np.argmax(y_test, axis=1)
    
    # Train and evaluate
    lr = LogisticRegression(C=0.1, max_iter=1000, random_state=42)
    lr.fit(X_train_flat, y_train_flat)
    
    train_acc = lr.score(X_train_flat, y_train_flat)
    val_acc = lr.score(X_val_flat, y_val_flat)
    test_acc = lr.score(X_test_flat, y_test_flat)
    
    print(f"Train acc: {train_acc:.3f} | Val acc: {val_acc:.3f} | Test acc: {test_acc:.3f}")
    record(eval_method, "Logistic Regression", train_acc, val_acc, test_acc)
    
    # Clean up memory
    del X_train, X_val, X_test, y_train, y_val, y_test


In [ ]:
# 2. RANDOM FOREST
print("\n=== RANDOM FOREST ===")

for eval_method in ['intra', 'inter']:
    # Load data
    if eval_method == 'intra':
        X_all, y_all = load_eeg_data(USE_SUBJECTS, CHANNELS_TO_USE, PRE_WINDOW_MS, POST_WINDOW_MS)
        X_train, X_val, X_test, y_train, y_val, y_test = prepare_datasets(X_all, y_all)
    else:
        X_train, X_val, X_test, y_train, y_val, y_test = load_data_inter()
    
    # Prepare for sklearn
    X_train_flat = X_train.reshape(X_train.shape[0], -1)
    X_val_flat = X_val.reshape(X_val.shape[0], -1)
    X_test_flat = X_test.reshape(X_test.shape[0], -1)
    y_train_flat = np.argmax(y_train, axis=1)
    y_val_flat = np.argmax(y_val, axis=1)
    y_test_flat = np.argmax(y_test, axis=1)
    
    # Train and evaluate
    rf = RandomForestClassifier(n_estimators=100, max_depth=10, random_state=42, n_jobs=-1)
    rf.fit(X_train_flat, y_train_flat)
    
    train_acc = rf.score(X_train_flat, y_train_flat)
    val_acc = rf.score(X_val_flat, y_val_flat)
    test_acc = rf.score(X_test_flat, y_test_flat)
    
    print(f"Train acc: {train_acc:.3f} | Val acc: {val_acc:.3f} | Test acc: {test_acc:.3f}")
    record(eval_method, "Random Forest", train_acc, val_acc, test_acc)
    
    # Clean up memory
    del X_train, X_val, X_test, y_train, y_val, y_test


In [ ]:
# 3. SIMPLE CNN
print("\n=== SIMPLE CNN ===")

for eval_method in ['intra', 'inter']:
    # Load data
    if eval_method == 'intra':
        X_all, y_all = load_eeg_data(USE_SUBJECTS, CHANNELS_TO_USE, PRE_WINDOW_MS, POST_WINDOW_MS)
        X_train, X_val, X_test, y_train, y_val, y_test = prepare_datasets(X_all, y_all)
    else:
        X_train, X_val, X_test, y_train, y_val, y_test = load_data_inter()
    
    Chans, Samples = X_train.shape[1], X_train.shape[2]
    
    # Build and train model
    model = Sequential([
        Conv2D(16, (1, 10), activation='relu', input_shape=(Chans, Samples, 1)),
        MaxPooling2D((1, 4)),
        Conv2D(32, (1, 5), activation='relu'),
        MaxPooling2D((1, 4)),
        Dropout(0.5),
        Flatten(),
        Dense(32, activation='relu'),
        Dense(2, activation='softmax')
    ])
    
    model.compile(optimizer='adam', loss='categorical_crossentropy', metrics=['accuracy'])
    
    # Track accuracy per epoch
    class AccHistory(tf.keras.callbacks.Callback):
        def on_train_begin(self, logs=None):
            self.train_acc = []
            self.val_acc = []
        def on_epoch_end(self, epoch, logs=None):
            self.train_acc.append(logs.get('accuracy'))
            self.val_acc.append(logs.get('val_accuracy'))
    acc_hist = AccHistory()
    early_stop = EarlyStopping(monitor='val_loss', patience=5, restore_best_weights=True)
    model.fit(X_train, y_train, validation_data=(X_val, y_val), 
              epochs=20, batch_size=32, callbacks=[early_stop, acc_hist], verbose=1)
    
    # Evaluate
    train_acc = model.evaluate(X_train, y_train, verbose=0)[1]
    val_acc = model.evaluate(X_val, y_val, verbose=0)[1]
    test_acc = model.evaluate(X_test, y_test, verbose=0)[1]
    
    print(f"Train acc: {train_acc:.3f} | Val acc: {val_acc:.3f} | Test acc: {test_acc:.3f}")
    print("Per-epoch train acc:", acc_hist.train_acc)
    print("Per-epoch val acc:", acc_hist.val_acc)
    record(eval_method, "Simple CNN", train_acc, val_acc, test_acc)
    
    # Clean up memory
    del X_train, X_val, X_test, y_train, y_val, y_test, model


In [ ]:
# 4. EEGNET
print("\n=== EEGNET ===")

for eval_method in ['intra', 'inter']:
    # Load data
    if eval_method == 'intra':
        X_all, y_all = load_eeg_data(USE_SUBJECTS, CHANNELS_TO_USE, PRE_WINDOW_MS, POST_WINDOW_MS)
        X_train, X_val, X_test, y_train, y_val, y_test = prepare_datasets(X_all, y_all)
    else:
        X_train, X_val, X_test, y_train, y_val, y_test = load_data_inter()
    
    Chans, Samples = X_train.shape[1], X_train.shape[2]
    
    # Build and train model
    model = EEGNet(nb_classes=2, Chans=Chans, Samples=Samples, 
                   dropoutRate=0.3, kernLength=64, F1=8, D=2, F2=16)
    model.compile(optimizer='adam', loss='categorical_crossentropy', metrics=['accuracy'])
    
    # Track accuracy per epoch
    class AccHistory(tf.keras.callbacks.Callback):
        def on_train_begin(self, logs=None):
            self.train_acc = []
            self.val_acc = []
        def on_epoch_end(self, epoch, logs=None):
            self.train_acc.append(logs.get('accuracy'))
            self.val_acc.append(logs.get('val_accuracy'))
    acc_hist = AccHistory()
    early_stop = EarlyStopping(monitor='val_loss', patience=8, restore_best_weights=True)
    model.fit(X_train, y_train, validation_data=(X_val, y_val),
              epochs=30, batch_size=32, callbacks=[early_stop, acc_hist], verbose=1)
    
    # Evaluate
    train_acc = model.evaluate(X_train, y_train, verbose=0)[1]
    val_acc = model.evaluate(X_val, y_val, verbose=0)[1]
    test_acc = model.evaluate(X_test, y_test, verbose=0)[1]
    
    print(f"Train acc: {train_acc:.3f} | Val acc: {val_acc:.3f} | Test acc: {test_acc:.3f}")
    print("Per-epoch train acc:", acc_hist.train_acc)
    print("Per-epoch val acc:", acc_hist.val_acc)
    record(eval_method, "EEGNet", train_acc, val_acc, test_acc)
    
    # Clean up memory
    del X_train, X_val, X_test, y_train, y_val, y_test, model


In [ ]:
# RESULTS COMPARISON
print("\n" + "="*70)
print("INTRA-SUBJECT vs INTER-SUBJECT COMPARISON")  
print("="*70)

for method in ['intra', 'inter']:
    print(f"\n{method.upper()}-SUBJECT RESULTS:")
    df = pd.DataFrame(results[method]).round(3)
    print(df.to_string(index=False))
    
    if len(df) > 0:
        avg_test = df['Test'].mean()
        avg_gap = df['Gap'].mean()
        best_model = df.loc[df['Test'].idxmax(), 'Model']
        print(f"Best: {best_model} | Avg Test: {avg_test:.1%} | Avg Gap: {avg_gap:.1%}")

# Compare methods
if len(results['intra']['Test']) > 0 and len(results['inter']['Test']) > 0:
    intra_avg = sum(results['intra']['Test']) / len(results['intra']['Test'])
    inter_avg = sum(results['inter']['Test']) / len(results['inter']['Test'])
    drop = intra_avg - inter_avg
    
    print(f"\n📊 SUMMARY:")
    print(f"Intra-subject avg: {intra_avg:.1%}")
    print(f"Inter-subject avg: {inter_avg:.1%}")
    print(f"Performance drop: {drop:.1%}")
    
    if drop > 0.1:
        print("⚠️  Large drop indicates severe overfitting in intra-subject evaluation")
    else:
        print("✅ Reasonable performance difference between methods")